## 1. Install Required Packages

In [ ]:
!pip install requests tqdm

## 2. Import Libraries

In [ ]:
import os
import requests
from tqdm import tqdm

## 3. Configuration

In [ ]:
# USA bounding box (includes Alaska and Hawaii)
USA_BBOX = {
    'lat_min': 18.91619,
    'lat_max': 71.3577635769,
    'lon_min': -171.791110603,
    'lon_max': -66.96466
}

# Output directory
OUTPUT_DIR = "USA_SoilGrids/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# SoilGrids soil properties (0-5cm depth, mean values)
SOIL_PROPERTIES = {
    'bdod': 'Bulk Density (cg/cm³)',
    'cec': 'Cation Exchange Capacity (mmol(c)/kg)',
    'cfvo': 'Coarse Fragments (cm³/dm³)',
    'clay': 'Clay Content (g/kg)',
    'nitrogen': 'Nitrogen (cg/kg)',
    'ocd': 'Organic Carbon Density (hg/m³)',
    'ocs': 'Organic Carbon Stocks (t/ha)',
    'phh2o': 'pH in H2O (pH×10)',
    'sand': 'Sand Content (g/kg)',
    'silt': 'Silt Content (g/kg)',
    'soc': 'Soil Organic Carbon (dg/kg)',
    'wv0010': 'Water Content at 10kPa (cm³/dm³)',
    'wv0033': 'Water Content at 33kPa (cm³/dm³)',
    'wv1500': 'Water Content at 1500kPa (cm³/dm³)',
}

# Depth layer to download
SELECTED_DEPTH = '0-5cm'

print(f"{'='*70}")
print(f"SoilGrids Download for USA")
print(f"{'='*70}")
print(f"Bounding Box: Lat [{USA_BBOX['lat_min']}, {USA_BBOX['lat_max']}]")
print(f"              Lon [{USA_BBOX['lon_min']}, {USA_BBOX['lon_max']}]")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Depth Layer: {SELECTED_DEPTH}")
print(f"Properties to download: {len(SOIL_PROPERTIES)}")
print(f"{'='*70}")

## 4. Download Function

In [ ]:
def download_soilgrids_property(property_name, description, depth='0-5cm'):
    """
    Download a single SoilGrids property for USA
    
    Args:
        property_name: Soil property code (e.g., 'clay', 'sand')
        description: Human-readable description
        depth: Depth layer (default '0-5cm')
    """
    
    # Construct WCS URL
    base_url = f"https://maps.isric.org/mapserv?map=/map/{property_name}.map"
    
    # Coverage ID format: property_depth_mean
    coverage_id = f"{property_name}_{depth}_mean"
    
    # WCS parameters
    params = {
        'SERVICE': 'WCS',
        'VERSION': '2.0.1',
        'REQUEST': 'GetCoverage',
        'COVERAGEID': coverage_id,
        'FORMAT': 'image/tiff',
        'SUBSET': [
            f"long({USA_BBOX['lon_min']},{USA_BBOX['lon_max']})",
            f"lat({USA_BBOX['lat_min']},{USA_BBOX['lat_max']})"
        ],
        'SUBSETTINGCRS': 'http://www.opengis.net/def/crs/EPSG/0/4326',
        'OUTPUTCRS': 'http://www.opengis.net/def/crs/EPSG/0/4326'
    }
    
    # Construct full URL
    url_parts = [base_url]
    for key, value in params.items():
        if key == 'SUBSET':
            for subset in value:
                url_parts.append(f"SUBSET={subset}")
        else:
            url_parts.append(f"{key}={value}")
    
    full_url = "&".join(url_parts)
    
    # Output filename
    output_file = os.path.join(OUTPUT_DIR, f"{property_name}_{depth}_mean.tif")
    
    # Check if already downloaded
    if os.path.exists(output_file):
        file_size = os.path.getsize(output_file) / (1024 * 1024)  # MB
        print(f"✓ Already exists: {property_name} ({file_size:.2f} MB)")
        return True
    
    print(f"\nDownloading: {property_name} - {description}")
    print(f"Coverage ID: {coverage_id}")
    
    try:
        # Send request with streaming
        response = requests.get(full_url, stream=True, timeout=300)
        
        if response.status_code == 200:
            # Get file size if available
            total_size = int(response.headers.get('content-length', 0))
            
            # Download with progress bar
            with open(output_file, 'wb') as f:
                if total_size > 0:
                    with tqdm(total=total_size, unit='B', unit_scale=True, 
                             desc=property_name) as pbar:
                        for chunk in response.iter_content(chunk_size=8192):
                            f.write(chunk)
                            pbar.update(len(chunk))
                else:
                    # No content-length header, download without progress
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
            
            file_size = os.path.getsize(output_file) / (1024 * 1024)  # MB
            print(f"✓ Downloaded: {output_file} ({file_size:.2f} MB)")
            return True
            
        else:
            print(f"✗ Failed: HTTP {response.status_code}")
            print(f"  Response: {response.text[:200]}")
            return False
            
    except requests.exceptions.Timeout:
        print(f"✗ Timeout: Request took longer than 5 minutes")
        return False
    except Exception as e:
        print(f"✗ Error: {e}")
        return False

## 5. Download All Soil Properties

In [ ]:
# Download all soil properties
successful = []
failed = []

total = len(SOIL_PROPERTIES)

for idx, (prop_name, description) in enumerate(SOIL_PROPERTIES.items(), 1):
    print(f"\n[{idx}/{total}] Processing: {prop_name}")
    print("-" * 70)
    
    success = download_soilgrids_property(prop_name, description, SELECTED_DEPTH)
    
    if success:
        successful.append(prop_name)
    else:
        failed.append(prop_name)

# Summary
print(f"\n{'='*70}")
print(f"DOWNLOAD SUMMARY")
print(f"{'='*70}")
print(f"✓ Successful: {len(successful)}/{total}")
print(f"✗ Failed: {len(failed)}/{total}")

if successful:
    print(f"\nSuccessfully downloaded:")
    for prop in successful:
        print(f"  ✓ {prop}")

if failed:
    print(f"\nFailed to download:")
    for prop in failed:
        print(f"  ✗ {prop}")

print(f"\nFiles saved to: {OUTPUT_DIR}")
print(f"{'='*70}")

## 6. Download Multiple Depths (Optional)

If you need multiple depth layers

In [ ]:
# Available depth layers in SoilGrids
DEPTH_LAYERS = ['0-5cm', '5-15cm', '15-30cm', '30-60cm', '60-100cm', '100-200cm']

# Select properties to download (or use all)
selected_props = ['clay', 'sand', 'silt', 'soc', 'phh2o']  # Example subset

print(f"Downloading {len(selected_props)} properties × {len(DEPTH_LAYERS)} depths = {len(selected_props) * len(DEPTH_LAYERS)} files")
print(f"{'='*70}\n")

for prop in selected_props:
    print(f"\n{'='*70}")
    print(f"Property: {prop} - {SOIL_PROPERTIES[prop]}")
    print(f"{'='*70}")
    
    for depth in DEPTH_LAYERS:
        download_soilgrids_property(prop, SOIL_PROPERTIES[prop], depth)

print(f"\n{'='*70}")
print("Multi-depth download complete!")
print(f"{'='*70}")